In [ ]:
from pathlib import Path
import re

import torch
from transformer.transformer import Transformer

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

step_checkpoints = []
for checkpoint_path in Path("weights").glob("BlockFormer_*_steps.pth"):
    match = re.fullmatch(r"BlockFormer_(\d+)_steps\.pth", checkpoint_path.name)
    if match:
        step_checkpoints.append((int(match.group(1)), checkpoint_path))

checkpoint_step, checkpoint_path = max(step_checkpoints)
model = torch.nn.DataParallel(Transformer(DEVICE)).to(DEVICE)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded {checkpoint_path} on {DEVICE}")

In [ ]:
input_string = "Once upon a time"

with torch.inference_mode():
    output = model.module.generate_output(input_string)

print(output)